In [7]:
import numpy as np
import matplotlib.pyplot as plt

%matplotlib ipympl

# Echauffement : le broadcasting
print('questions concernant a')

a = np.array([1, 2, 3])
# 1. visualisation de l'action du None
print(a, a.shape)
print(a[:, None], a[:, None].shape)
print(a[None, :], a[None, :].shape)

# 2. forme du tableau X : (3,3) ; X[i,j]=i-j
X=a[:, None]-a[None, :]
print(X, X.shape)

# 3. 
print('questions concernant b')
# forme de b[:, None, :] -> (2,1,3)
# forme de b[:, :, None] -> (2,3,1)
# donc forme du tableau calculé : (2,3,3)
b=np.array([[1,2,3],[4,5,6]])
print(b, b.shape)
Y= b[:, None, :] - b[:, :, None]
print(Y,Y.shape)

questions concernant a
[1 2 3] (3,)
[[1]
 [2]
 [3]] (3, 1)
[[1 2 3]] (1, 3)
[[ 0 -1 -2]
 [ 1  0 -1]
 [ 2  1  0]] (3, 3)
questions concernant b
[[1 2 3]
 [4 5 6]] (2, 3)
[[[ 0  1  2]
  [-1  0  1]
  [-2 -1  0]]

 [[ 0  1  2]
  [-1  0  1]
  [-2 -1  0]]] (2, 3, 3)


In [ ]:
# Initialisation aléatoire

# les bornes pour le tirage au sort initial
mass_max = 3.
    
x_min, x_max = -10., 10.
y_min, y_max = -10., 10.

speed_max = 1.
# # # # #

def init_problem(N):
    """
    retourne un tuple masses, positions, speeds
    de formes resp.   (N,)    (2, N)     (2, N)
    tiré au sort dans les bornes définies ci-dessus
    """
    masses=np.random.uniform(0,mass_max,(N,))

    positionsx, positionsy =  np.random.uniform(x_min,x_max,(N,)), np.random.uniform(y_min,y_max,(N,))
    positionsx, positionsy = positionsx[None, :], positionsy[None, :] # empaquetage de chaque ligne pour avoir 2 lignes différentes lors de la concaténation
    positions = np.concatenate((positionsx, positionsy), axis=0)

    speedsx, speedsy =  np.random.uniform(0,speed_max,(N,)), np.random.uniform(0,speed_max,(N,))
    speedsx, speedsy=speedsx[None, :], speedsy[None, :]
    speeds = np.concatenate((speedsx, speedsy), axis=0)

    return masses, positions, speeds

# pour tester

# normalement vous devez pouvoir faire ceci

masses, positions, speeds = init_problem(10)

#print(masses, masses.shape)
#print(positions, positions.shape)
#print(speeds, speeds.shape)

# et ceci devrait afficher OK
try:
    assert masses.shape == (10,) and positions.shape == speeds.shape == (2, 10)
    print("OK")
except:
    print("KO")

[2.76311288 2.34568548 2.18098956 0.94590384 1.63382474 2.01363926
 1.81957417 2.89786253 1.95267068 1.9712495 ] (10,)
[[ 5.17091684 -8.4053363  -7.18034587 -8.31565117  3.11922746 -3.96744211
   6.18888338 -2.33649922  5.79811286  1.91341262]
 [ 4.24798955  2.02190671 -3.44155386 -5.64579869  4.10696703  8.8165398
  -7.94708582 -5.45026905 -3.46264362  5.98561422]] (2, 10)
[[0.19292861 0.69222742 0.63890447 0.62261137 0.07273232 0.7720503
  0.75938626 0.9857686  0.18298974 0.31233813]
 [0.88593452 0.85374635 0.80784591 0.35644495 0.18012852 0.42009969
  0.19422797 0.77014707 0.9611333  0.65579368]] (2, 10)
OK


In [18]:
# Initialisation reproductible

def init3():
    # first element is sun-like: heavy, at the center, and no speed
    masses = np.array([3, 1, 1], dtype=float)
    positions = np.array([
        [0, 5, -5], 
        [0, 1, -1]], dtype=float)
    speeds = np.array([
        [0, -1, 1], 
        [0, 0, 0]], dtype=float)
    return masses, positions, speeds

In [ ]:
# Les forces

def forces(masses, positions, G=1.0):
    """
    returns an array of shape (2,  N)
    that contains the force felt by each mass from all the others
    G is the Newton constant and by default is set to 1 
    (we have abstract units anyway)
    """
    # calcul des vecteurs r_j-r_i puis de leur norme
    diff= positions[:, None, :] - positions[:, :, None] #c'est le sens pour lequel diff[:,i,j] est effectivement le vecteur r_j-r_i quand on teste avec les valeurs connues
    #print(diff)
    dist = np.linalg.norm(diff, axis=0)
    np.fill_diagonal(dist, np.inf) # pour que les termes diagonaux de coeff soient nuls

    # calcul des termes de la somme (puis somme)
    pdtmasses= masses[:, None] * masses[None, :]
    coeff= G * pdtmasses / dist**3
    termes_vect= coeff[None, :, :] * diff
    ls_forces = np.sum(termes_vect, axis=2)

    return ls_forces

# pour tester, voici les valeurs attendues avec la config prédéfinie

masses, positions, speeds = init3()

#print(positions)

f = forces(masses, positions)

# should be true
print( np.all(np.isclose(f, np.array([
     [ 0.        , -0.12257258,  0.12257258],
     [ 0.        , -0.02451452,  0.02451452]]))) )

[[[  0.   5.  -5.]
  [ -5.   0. -10.]
  [  5.  10.   0.]]

 [[  0.   1.  -1.]
  [ -1.   0.  -2.]
  [  1.   2.   0.]]]
[[ 0.  5. -5.]
 [ 0.  1. -1.]]
True


In [31]:
# Le simulateur

def simulate(masses, positions, speeds, dt=0.1, nb_steps=100):
    """
    should return the positions across time
    so an array of shape (nb_steps, 2, N)
    optional dt is the time step
    """
    #initialisation des tableaux stockant les positions au cours du temps
    dim, N = positions.shape
    trajectoires=np.zeros((nb_steps, dim, N))
    trajectoires[0, :, :]=positions
    speeds2=np.copy(speeds) # une copie du tableau passé en paramètre qui pourra être modifiée par la suite

    for step in range(1, nb_steps): #calcul de proche en proche des positions au cours du temps
        ls_forces=forces(masses, trajectoires[step-1, :, :])
        acc = ls_forces / masses[None, :]
        speeds2 += dt*acc
        trajectoires[step, :, :] = trajectoires[step-1, :, :] + dt*speeds2

    return trajectoires

# pour tester la forme

SMALL_STEPS = 4

s = simulate(masses, positions, speeds, nb_steps=SMALL_STEPS)

try:
    if s.shape == (SMALL_STEPS, 2, 3):
        print("shape OK")
except Exception as exc:
    print(f"OOPS {type(exc)} {exc}")

# pour tester la trajectoire (premier point seulement)

positions1 = s[1]

print( np.all(np.isclose(positions1, np.array([
    [ 0.        ,  4.89877427, -4.89877427],
    [ 0.        ,  0.99975485, -0.99975485]]))) )

[[[  0.   5.  -5.]
  [ -5.   0. -10.]
  [  5.  10.   0.]]

 [[  0.   1.  -1.]
  [ -1.   0.  -2.]
  [  1.   2.   0.]]]
[[[ 0.          4.89877427 -4.89877427]
  [-4.89877427  0.         -9.79754855]
  [ 4.89877427  9.79754855  0.        ]]

 [[ 0.          0.99975485 -0.99975485]
  [-0.99975485  0.         -1.99950971]
  [ 0.99975485  1.99950971  0.        ]]]
[[[ 0.          4.79627468 -4.79627468]
  [-4.79627468  0.         -9.59254935]
  [ 4.79627468  9.59254935  0.        ]]

 [[ 0.          0.99924973 -0.99924973]
  [-0.99924973  0.         -1.99849947]
  [ 0.99924973  1.99849947  0.        ]]]
shape OK
True
